# THERMAL MODEL RUN 

## Dry Regolith

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from Regolith1D_JGR import Regolith1D, Grid1D
import pandas as pd

In [2]:
z_max = 10
N = 50 # Number of intervals → N+1 nodes
rego_grid = Grid1D(N, z_max =z_max)

alpha = 6
z, dz = rego_grid.create_grid_exp2(alpha = alpha)

In [3]:
latitude_vector = np.arange(0, 81, 10)

In [4]:
def write_file(parent_folder, T_hist, lat, dt):
    # -----------------------------
    # TIME SETUP
    # -----------------------------
    T_period = 24 * 3600
    steps_per_day = int(T_period / dt)

    if not np.isclose(T_period / dt, steps_per_day):
        raise ValueError("T_period/dt must be an integer.")

    local_time_seconds = np.arange(steps_per_day) * dt  # rows

    # -----------------------------
    # LOOP OVER LATITUDES
    # -----------------------------
    print(f"Processing latitude = {lat}°")

    # -----------------------------
    # INIT ARRAYS
    # -----------------------------
    T_avg = np.zeros((steps_per_day, T_hist.shape[1]))  # (time, depth)
    counts = np.zeros(steps_per_day)

    # -----------------------------
    # PHASE AVERAGING
    # -----------------------------
    for k in range(n_steps + 1):
        phase = k % steps_per_day
        T_avg[phase, :] += T_hist[k, :]
        counts[phase] += 1

    # Normalize
    valid = counts > 0
    T_avg[valid, :] /= counts[valid][:, None]

    # -----------------------------
    # BUILD DATAFRAME
    # -----------------------------
    df = pd.DataFrame(T_avg, columns=[f"{z[j]:.5f}" for j in range(T_hist.shape[1])])

    # Add local time as first column
    df.insert(0, "local_time_s", local_time_seconds)

    # Round temperatures
    df.iloc[:, 1:] = df.iloc[:, 1:].round(3)

    # -----------------------------
    # SAVE
    # -----------------------------
    filename = f"T_profile_lat{lat}.csv"
    filepath = os.path.join(parent_folder, filename)

    df.to_csv(filepath, index=False)

In [ ]:
n_days = 62 # Days in local time (lunar). For trial use 1 to reduce computation time
day_s = 86400
max_time = n_days * day_s
n_steps = int(n_days * 1440)
dt = max_time / (n_steps)

time = np.linspace(0, max_time, n_steps+1)
local_time = np.linspace(0, max_time, n_steps+1)/3600

In [6]:
import os
# Choose the folder where you want to store results
parent_folder = os.path.join(os.getcwd(), "FIGS", "1D", f"{n_days}localdays_dt{dt}_N{N}_alpha{alpha}_zmax{z_max}")
os.makedirs(parent_folder, exist_ok=True)

In [7]:
T_profile_lat = []

for lam in latitude_vector:
    print(f"Calculating T profile for lat = {lam} º")
    T_history = Regolith1D(N=N, lmbda=lam, z=z, dz=dz, t_max = max_time, n_steps = n_steps).calculate_temperature_profile_explicit()
    write_file(parent_folder, T_history, lam, dt)
    T_profile_lat.append(T_history)

Calculating T profile for lat = 0 º
0.0% time steps computed
25.0% time steps computed
50.0% time steps computed
75.0% time steps computed
Processing latitude = 0°
Calculating T profile for lat = 10 º
0.0% time steps computed
25.0% time steps computed
50.0% time steps computed
75.0% time steps computed
Processing latitude = 10°
Calculating T profile for lat = 20 º
0.0% time steps computed
25.0% time steps computed
50.0% time steps computed
75.0% time steps computed
Processing latitude = 20°
Calculating T profile for lat = 30 º
0.0% time steps computed
25.0% time steps computed
50.0% time steps computed
75.0% time steps computed
Processing latitude = 30°
Calculating T profile for lat = 40 º
0.0% time steps computed
25.0% time steps computed
50.0% time steps computed
75.0% time steps computed
Processing latitude = 40°
Calculating T profile for lat = 50 º
0.0% time steps computed
25.0% time steps computed
50.0% time steps computed
75.0% time steps computed
Processing latitude = 50°
Calcul

## Icy Regolith

In [8]:
lams = [85, 86, 87, 88, 89, 90]
wts = [5]
z0_ices = 0.01

In [9]:
N = 25 # Number of intervals → N+1 nodes
z_max = 0.5
alpha = 3
rego_grid = Grid1D(N, z_max = z_max)
z, dz = rego_grid.create_grid_exp2(alpha = alpha)

In [ ]:
n_days = 62 # 62 local days = 5x365 earth days
day_s = 86400
max_time = n_days * day_s
n_steps = int(n_days * 1440)

time = np.linspace(0, max_time, n_steps+1)
local_time = np.linspace(0, max_time, n_steps+1)/3600

In [11]:
epsilon = 0.95

In [12]:
def write_file_water(parent_folder, T_hist, wt, z0_ice, dt):
    # -----------------------------
    # TIME SETUP
    # -----------------------------
    T_period = 24 * 3600
    steps_per_day = int(T_period / dt)

    if not np.isclose(T_period / dt, steps_per_day):
        raise ValueError("T_period/dt must be an integer.")

    local_time_seconds = np.arange(steps_per_day) * dt  # rows

    # -----------------------------
    # LOOP OVER LATITUDES
    # -----------------------------
    print(f"Processing wt = {wt}%, zo = {z0_ice}")

    # -----------------------------
    # INIT ARRAYS
    # -----------------------------
    T_avg = np.zeros((steps_per_day, T_hist.shape[1]))  # (time, depth)
    counts = np.zeros(steps_per_day)

    # -----------------------------
    # PHASE AVERAGING
    # -----------------------------
    for k in range(n_steps + 1):
        phase = k % steps_per_day
        T_avg[phase, :] += T_hist[k, :]
        counts[phase] += 1

    # Normalize
    valid = counts > 0
    T_avg[valid, :] /= counts[valid][:, None]

    # -----------------------------
    # BUILD DATAFRAME
    # -----------------------------
    df = pd.DataFrame(T_avg, columns=[f"{z[j]:.5f}" for j in range(T_hist.shape[1])])

    # Add local time as first column
    df.insert(0, "local_time_s", local_time_seconds)

    # Round temperatures
    df.iloc[:, 1:] = df.iloc[:, 1:].round(3)

    # -----------------------------
    # SAVE
    # -----------------------------
    filename = f"T_profile_lat{lam}_wt{wt}_z0_ice{z0_ice}.csv"
    filepath = os.path.join(parent_folder, filename)

    df.to_csv(filepath, index=False)

In [16]:
import os
dt = max_time / (n_steps)
parent_folder_sim = os.path.join(os.getcwd(), "FIGS", "1D_icy", f"{n_days}localdays_dt{dt}_N{N}_zmax{z_max}_alpha{alpha}_eps_{0.95}")
os.makedirs(parent_folder_sim, exist_ok=True)

In [17]:
T_profile_lat = []

for lam in lams:
    for wt in wts:
        print(f"Calculating T profile for lam = {lam}º, wt = {wt} %, ice depth = {z0_ices}")
        T_history = Regolith1D(N=N, lmbda=lam, z=z, dz=dz, epsilon = epsilon, t_max = max_time, n_steps = n_steps, wt = wt, z0_ice = z0_ices).calculate_temperature_profile_explicit()
        write_file_water(parent_folder_sim, T_history, wt, z0_ices, dt)
        T_profile_lat.append(T_history)

Calculating T profile for lam = 85º, wt = 5 %, ice depth = 0.01
0.0% time steps computed
25.0% time steps computed
50.0% time steps computed
75.0% time steps computed
Processing wt = 5%, zo = 0.01
Calculating T profile for lam = 86º, wt = 5 %, ice depth = 0.01
0.0% time steps computed
25.0% time steps computed
50.0% time steps computed
75.0% time steps computed
Processing wt = 5%, zo = 0.01
Calculating T profile for lam = 87º, wt = 5 %, ice depth = 0.01
0.0% time steps computed
25.0% time steps computed
50.0% time steps computed
75.0% time steps computed
Processing wt = 5%, zo = 0.01
Calculating T profile for lam = 88º, wt = 5 %, ice depth = 0.01
0.0% time steps computed
25.0% time steps computed
50.0% time steps computed
75.0% time steps computed
Processing wt = 5%, zo = 0.01
Calculating T profile for lam = 89º, wt = 5 %, ice depth = 0.01
0.0% time steps computed
25.0% time steps computed
50.0% time steps computed
75.0% time steps computed
Processing wt = 5%, zo = 0.01
Calculating T p